In [ ]:
from parmed.amber import NetCDFTraj, AmberParm
import numpy as np
import math

In [ ]:
def dij(x1, x2):
    v = x2 - x1
    r = np.linalg.norm(v)
    return r

def dij_inv(x1, x2):
    v = x2 - x1
    r = np.linalg.norm(v)
    return  1 / r

def lennard_jones_energy(x1, x2, epsilon, sigma):
    v = x2 - x1
    r = np.linalg.norm(v)
    e = 4 * epsilon * ((sigma / r)**12 - (sigma / r)**6)
    f = ((4 * epsilon) / r) * (-12 * (sigma / r)**12 + 6 * (sigma / r)**6) * (v / r)
    return e, f

In [ ]:
parm = AmberParm('../input/step3_pbcsetup.parm7')

# NEED TO EXCLUDE BONDED INTERACTIONS
nonbonded_exclusion = [set() for _ in parm.atoms]

for bond in parm.bonds:
    nonbonded_exclusion[bond.atom1.idx].add(bond.atom2.idx)
    nonbonded_exclusion[bond.atom2.idx].add(bond.atom1.idx)

for angle in parm.angles:
    nonbonded_exclusion[angle.atom1.idx].add(angle.atom3.idx)
    nonbonded_exclusion[angle.atom3.idx].add(angle.atom1.idx)

for dihedral in parm.dihedrals:
    nonbonded_exclusion[dihedral.atom1.idx].add(dihedral.atom4.idx)
    nonbonded_exclusion[dihedral.atom4.idx].add(dihedral.atom1.idx)    

In [ ]:
n_windows = 1

pos_all = []
for i in range(n_windows):
    pos = []
    traj = NetCDFTraj.open_old("../%02d/step7_reprocessing.nc" % i)
    pos.append(traj.coordinates)
    traj.close()
    pos_all.append(np.concatenate(pos)[::250])

In [ ]:
qm_atoms = np.array([26638,26639,27261], dtype=int) - 1

prot_forces = []
near_forces = []

for window in range(len(pos_all)):
    for frame in range(len(pos_all[window])):
        forces = np.zeros((len(qm_atoms), 3, len(parm.atoms)))
        for i, atom in enumerate(qm_atoms):
            atom1 = pos_all[window][frame][atom]
            for j, atoms in enumerate(pos_all[window][frame]):   
                atom2 = pos_all[window][frame][j]
                if atom != j and dij(atom1, atom2) <= 10 and not j in nonbonded_exclusion[atom]:
                    epsilon = np.sqrt(parm.atoms[atom].epsilon * parm.atoms[j].epsilon)
                    sigma = (parm.atoms[atom].sigma + parm.atoms[j].sigma) / 2.0
                    e, f = lennard_jones_energy(atom1, atom2, epsilon, sigma)
                    forces[i, :, j] = f
    prot_forces.append(forces)
    near_forces.append(forces.sum(axis=2))
                